# Gradient Extraction & Storing

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

#########################################
# 1. Define a simple CNN with 3 conv layers
#########################################
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # conv1: from 4x4 input → output remains 4x4
        self.conv1 = nn.Conv2d(1, 1, kernel_size=3, stride=1, padding=1)
        # conv2: from 4x4 → 2x2 (stride=2)
        self.conv2 = nn.Conv2d(1, 1, kernel_size=3, stride=1, padding=1)
        # conv3: from 2x2 → 1x1 (stride=2)
        self.conv3 = nn.Conv2d(1, 1, kernel_size=3, stride=1, padding=1)
        self.fcn = nn.Linear(16, 10)  # not used for interaction

    def forward(self, x):
        a1 = torch.tanh(self.conv1(x))    # shape: (1,1,4,4)
        a2 = torch.tanh(self.conv2(a1))     # shape: (1,1,2,2)
        a3 = torch.tanh(self.conv3(a2))     # shape: (1,1,1,1)
        return self.fcn(a3.reshape(1, -1))

model = SimpleCNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)


#########################################
# 2. Register hooks to capture gradients
#########################################
activation_gradients = {}
gradient_flows = {}
def get_activation_grad(name, connet2name=None):
    def hook(module, grad_input, grad_output):
        if name is not None:
            # Lấy grad_out: shape [N, C_out, H_out, W_out]
            grad_out = grad_output[0].detach()
            N, C_out, H_out, W_out = grad_out.shape
            
            # Lấy weight của module: shape [C_out, C_in, kH, kW]
            weight = module.weight  
            C_out_w, C_in, kH, kW = weight.shape
            assert C_out == C_out_w, "Mismatch in output channels."

            # Chuyển weight thành dạng ma trận: [C_out, C_in*kH*kW]
            weight_reshaped = weight.view(C_out, -1)
            
            # Reshape grad_out thành [N, C_out, Len_out] với Len_out = H_out * W_out
            Len_out = H_out * W_out
            grad_out_reshaped = grad_out.view(N, C_out, Len_out)

            # Tính grad_input_cols: [N, C_in*kH*kW, Len_out]
            grad_input_cols = torch.matmul(weight_reshaped.t(), grad_out_reshaped)
            
            # Giả sử batch size N=1
            grad_input_cols = grad_input_cols[0]  # [C_in*kH*kW, Len_out]

            # Lấy kích thước input từ grad_input[0]: [N, C_in, H_in, W_in]
            H_in, W_in = grad_input[0].shape[2:]
            Len_in = H_in * W_in

            # Xây dựng ánh xạ từ các patch đến các vị trí trên input:
            # Tạo tensor chứa các chỉ số của các ô input, shape: [1, 1, H_in, W_in]
            input_indices = torch.arange(Len_in, device=grad_out.device).view(1, 1, H_in, W_in).float()
            # Sử dụng F.unfold để lấy ma trận ánh xạ, shape: [C_in*kH*kW, Len_out]
            idx_map = F.unfold(input_indices, kernel_size=module.kernel_size,
                               dilation=module.dilation, padding=module.padding, stride=module.stride)[0]

            # Khởi tạo gradient_flows với kích thước (Len_in, Len_out)
            gradient_flow = torch.zeros(Len_in, Len_out, device=grad_out.device)
            # Sử dụng scatter_add_ để cộng các giá trị từ grad_input_cols vào gradient_flows
            # Cho mỗi phần tử tại vị trí (p, j) trong grad_input_cols, ta cộng vào gradient_flows tại (idx_map[p,j], j)
            gradient_flow.scatter_add_(0, idx_map.long(), grad_input_cols)

            gradient_flow = torch.abs(gradient_flow)
            gradient_flow[gradient_flow>1e-5] = 1.
            gradient_flow[gradient_flow<=1e-5] = .99
            # Lưu kết quả vào activation_gradients
            gradient_flows[(name, connet2name)] = gradient_flow.cpu().numpy()  # kích thước: (Len_in, Len_out)
            
            activation_gradients[name] = (gradient_flows[(name, connet2name)]).sum(axis=-1,keepdims=False).reshape(H_in, W_in)
            activation_gradients[connet2name] = (gradient_flows[(name, connet2name)]).sum(axis=0,keepdims=False).reshape(H_out, W_out)
    return hook

model.conv1.register_backward_hook(get_activation_grad(None, "conv1"))
model.conv2.register_backward_hook(get_activation_grad("conv1", "conv2"))
model.conv3.register_backward_hook(get_activation_grad("conv2", "conv3"))

#########################################
# 3. Run forward/backward on a random input
#########################################
input_tensor = torch.randn(1, 1, 4, 4)
label = torch.tensor([1])
optimizer.zero_grad()
a3 = model(input_tensor)
loss = criterion(a3.reshape(1, -1), label)  # use conv3 output for loss
loss.backward()

Save gradient flow information, includes: 
- activation_gradients: dict of aggregated weights of nodes at each layer. 
- gradient_flows: dict of values of gradient flows between adjacent layers.

In [3]:
import pickle
flow_info = {"activation_gradients": activation_gradients, 
             "gradient_flows": gradient_flows}
with open('flow_info.pkl', 'wb') as f:
    pickle.dump(flow_info, f)

# Rendering the Visualization

In [3]:
import numpy as np
from torchvision import transforms
import plotly.graph_objs as go
import ipywidgets as widgets
from IPython.display import display
from matplotlib import cm as cm
from matplotlib import colors as mcolors

import pickle
with open('flow_info.pkl', 'rb') as f:
    flow_info = pickle.load(f)
activation_gradients = (flow_info["activation_gradients"])
gradient_flows = (flow_info["gradient_flows"])

#########################################
# 4. Process gradients: obtain heatmaps and flattened vectors
#########################################
grad_vecs = dict()
node_layers, selectable_layers = [], []
for key, value in activation_gradients.items():
    node_layers.append(key)
    if type(key) is not tuple:
        grad_vecs[key] = value.reshape(-1)
for key, value in gradient_flows.items():
    node_layers.append(key[0])
    node_layers.append(key[1])
    if (key[0]) not in grad_vecs:
        grad_vecs[key[0]] = value.sum(axis=-1,keepdims=True)
    selectable_layers.append(key[0])
    if (key[1]) not in grad_vecs:
        grad_vecs[key[1]] = value.sum(axis=0,keepdims=True)
node_layers = list(sorted(set(node_layers)))
selectable_layers = list(sorted(set(selectable_layers)))

#########################################
# 5. Compute flow matrices via outer product, with thresholding
#########################################
threshold = 0.0
def compute_flow(gradient_flows, grad_vecs, threshold=None):
    flow_matrices = dict()
    for source, source_grad in grad_vecs.items():
        for target, target_grad in grad_vecs.items():
            key = (source, target)
            if key in gradient_flows:
                value = gradient_flows[(source, target)]
                if value is None:
                    value = source_grad.reshape(-1,1)/target_grad.reshape(1,-1)
                    value[np.isnan(value)] = 0
                    value = value/np.sum(value, axis=-1, keepdims=True)
                    value[np.isnan(value)] = 0
                flow_matrices[key] = value*1.
                if threshold is not None:
                    (flow_matrices[key])[value<threshold] = 0
    return flow_matrices
flow_matrices = compute_flow(gradient_flows, grad_vecs)

# Determine dynamic threshold slider limits:
flow_slider_min, flow_slider_max = None, None
for key, value in flow_matrices.items():
    if (flow_slider_min is None) or (value.min()<flow_slider_min):
        flow_slider_min = value.min()
    if (flow_slider_max is None) or (value.max()>flow_slider_max):
        flow_slider_max = value.max()

#########################################
# 6. Build Sankey data for layers. 
#########################################
global_index = {}  # Mapping: key = (layer, (row, col))
node_labels = []
current_index = 0

# Use distinct colormaps for nodes:
colors = dict()
color_coin = 0
node_colors = list()
for key in node_layers:
    value = (grad_vecs[key])
    color_list = None
    if color_coin%2:
        color_list = [mcolors.to_hex(cm.viridis(i/len(value))) for i in range(len(value))]
    else:
        color_list = [mcolors.to_hex(cm.plasma(i/len(value))) for i in range(len(value))]
    colors[key] = color_list
    color_coin += 1
    node_colors = node_colors + color_list
node_orders = dict()
node_x, node_y = [], []
for i, label in enumerate(node_layers):
    node_orders[label] = [(r, c) for r in range((activation_gradients[label]).shape[-2]) for c in range((activation_gradients[label]).shape[-1])]
    for j, (r, c) in enumerate(node_orders[label]):
        node_labels.append(f"{label}_{r},{c}")
        node_x.append((float(i)-.05)/(len(node_layers)-1.))
        node_y.append((float(j)-.05)/(len(node_orders[label])-1.))
        global_index[(label, (r, c))] = current_index
        current_index += 1

# Assign explicit node positions to match heatmap ordering.


#########################################
# 7. Create functions to generate Plotly figures (Sankey & Heatmaps)
#########################################
def create_sankey(selected_layer, selected_index, threshold):
    global flow_matrices
    # Recompute flows with current threshold.
    del flow_matrices
    flow_matrices = compute_flow(gradient_flows, grad_vecs, threshold=threshold)
    new_sources, new_targets, new_values = [], [], []
    for key, value in flow_matrices.items():
        source, target = key
        source_order = (node_orders[source])
        target_order = (node_orders[target])
        for i, (r1, c1) in enumerate(source_order):
            for j, (r2, c2) in enumerate(target_order):
                v = value[i, j]
                if v > 0:
                    new_sources.append(global_index[(source, (r1, c1))])
                    new_targets.append(global_index[(target, (r2, c2))])
                    new_values.append(v)
    
    new_node_colors, new_link_colors = [], []
    for key in node_layers:
        new_node_color = (colors[key])
        if selected_layer == key:
            new_node_color = [("red" if i == selected_index else new_node_color[i]) for i in range(len(grad_vecs[selected_layer]))]
        new_node_colors = new_node_colors + new_node_color
    for vi, src in enumerate(new_sources):
        new_link_colors.append("red" if (src == global_index[(selected_layer, (node_orders[selected_layer])[selected_index])]) and (float(new_values[vi])>0.) else f"rgba({128*int(float(new_values[vi])>0.)},{128*int(float(new_values[vi])>0.)},{128*int(float(new_values[vi])>0.)},.1)")

    fig = go.Figure(data=[go.Sankey(
        arrangement='fixed',
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color='black', width=0.5),
            label=node_labels,
            color=new_node_colors,
            x=node_x,
            y=node_y
        ),
        link=dict(
            source=new_sources,
            target=new_targets,
            value=new_values,
            color=new_link_colors
        )
    )])
    fig.update_layout(title_text='Gradient Flow Sankey Diagram', font_size=10)
    return fig

def create_heatmap_conv1(selected_index, selected_layer):
    shapes = []
    if selected_index is not None:
        row, col = divmod(selected_index, (activation_gradients[selected_layer]).shape[-1])
        shapes.append(dict(
            type='circle', xref='x', yref='y',
            x0=col - 0.3, y0=row - 0.3, x1=col + 0.3, y1=row + 0.3,
            line=dict(color='red', width=3)
        ))
    fig = go.Figure(data=go.Heatmap(
        z=activation_gradients[selected_layer],
        colorscale='Viridis',
        zmin=(activation_gradients[selected_layer]).min(),
        zmax=(activation_gradients[selected_layer]).max(),
        colorbar=dict(title='Selected-Layer Gradients')
    ))
    fig.update_layout(title='Selected-Layer Gradient', shapes=shapes,
                      xaxis=dict(dtick=1), yaxis=dict(dtick=1))
    return fig

def create_heatmap_conv2(selected_index, selected_layer):
    shapes = []
    connected_layer = selected_layer
    for key in gradient_flows.keys():
        if (key[0])==selected_layer:
            connected_layer = (key[1])
    grid_size = (activation_gradients[selected_layer]).shape
    flow_matrix = ((flow_matrices[(selected_layer, connected_layer)]>0.)[selected_index,:]).reshape(*grid_size)
    for r in range(grid_size[0]):
        for c in range(grid_size[1]):
            if flow_matrix[r,c] > 0.:
                shapes.append(dict(
                    type='circle', xref='x', yref='y',
                    x0=c - 0.3, y0=r - 0.3, x1=c + 0.3, y1=r + 0.3,
                    line=dict(color='red', width=3)
                ))
    fig = go.Figure(data=go.Heatmap(
        z=activation_gradients[connected_layer],
        colorscale='Plasma',
        zmin=(activation_gradients[connected_layer]).min(),
        zmax=(activation_gradients[connected_layer]).max(),
        colorbar=dict(title='Output-Layer Gradients')
    ))
    fig.update_layout(title='Output-Layer Gradient', shapes=shapes,
                      xaxis=dict(dtick=1), yaxis=dict(dtick=1))
    return fig

def create_heatmap(selected_index, selected_layer):
    fig1 = create_heatmap_conv1(selected_index, selected_layer)
    fig2 = create_heatmap_conv2(selected_index, selected_layer)
    return fig1, fig2


#########################################
# 8. IPyWidgets for Interaction: Dropdown for layer, Slider for node, and Slider for threshold.
#########################################
layer_dropdown = widgets.Dropdown(
    options=selectable_layers,
    value=selectable_layers[0],
    description='Select Layer:'
)

node_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=(np.cumprod(activation_gradients[layer_dropdown.value].shape)[-1])-1,
    step=1,
    description='Node Index:'
)

threshold_slider = widgets.FloatSlider(
    value=flow_slider_max,
    min=flow_slider_min,
    max=flow_slider_max,
    step=(flow_slider_max - flow_slider_min)/100,
    description='Threshold:'
)

def update_slider_range(selected_layer):
    node_slider.min = 0
    node_slider.max = (np.cumprod(activation_gradients[selected_layer].shape)[-1])-1
    if node_slider.value > node_slider.max:
        node_slider.value = 0

#########################################
# 9. Update figures based on interactions
#########################################
def update_figures(selected_layer, selected_index, threshold):
    sankey_fig_updated = create_sankey(selected_layer, selected_index, threshold)
    heatmap1_updated, heatmap2_updated = create_heatmap(selected_index, selected_layer)
    return sankey_fig_updated, heatmap1_updated, heatmap2_updated

output_sankey = widgets.Output()
output_conv1 = widgets.Output()
output_conv2 = widgets.Output()

def on_interaction_change(change):
    layer = layer_dropdown.value
    node = node_slider.value
    thresh = threshold_slider.value
    update_slider_range(layer)
    fig1, fig2, fig3 = update_figures(layer, node, thresh)
    with output_sankey:
        output_sankey.clear_output(wait=True)
        display(fig1)
    with output_conv1:
        output_conv1.clear_output(wait=True)
        display(fig2)
    with output_conv2:
        output_conv2.clear_output(wait=True)
        display(fig3)

layer_dropdown.observe(on_interaction_change, names='value')
node_slider.observe(on_interaction_change, names='value')
threshold_slider.observe(on_interaction_change, names='value')

ui = widgets.VBox([layer_dropdown, node_slider, threshold_slider, 
                   output_sankey, output_conv1, output_conv2])
display(ui)
on_interaction_change({'new': node_slider.value})